# 04 Early-Warning Label Creation

This notebook creates early-warning labels for the MetroPT-3 predictive maintenance framework.

It uses the feature-engineered dataset from:

`data/processed/metropt3_features.csv`

It creates:

- `warning_6h`
- `warning_12h`
- `warning_24h`
- `failure_interval`
- `model_include`

The main modelling target will be:

`warning_12h`

Actual failure intervals are excluded from normal modelling records using `model_include = False`.

In [ ]:
# 1. Import libraries and confirm environment

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Python executable:", sys.executable)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 1. Set project paths

In [ ]:
# 2. Define project paths

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

features_path = PROCESSED_DATA_DIR / "metropt3_features.csv"
labelled_path = PROCESSED_DATA_DIR / "metropt3_labelled.csv"
model_ready_path = PROCESSED_DATA_DIR / "metropt3_model_ready_12h.csv"

print("Project root:", PROJECT_ROOT)
print("Feature dataset path:", features_path)
print("Labelled output path:", labelled_path)
print("Model-ready output path:", model_ready_path)

## 2. Load feature-engineered dataset

This notebook must be run after `03_feature_engineering.ipynb`.

In [ ]:
# 3. Load feature-engineered dataset

if not features_path.exists():
    raise FileNotFoundError(
        f"Feature-engineered dataset not found at {features_path}. "
        "Run 03_feature_engineering.ipynb first."
    )

df = pd.read_csv(features_path)

print("Loaded feature-engineered dataset.")
print("Shape:", df.shape)
df.head()

## 3. Detect timestamp column and sort chronologically

In [ ]:
# 4. Detect timestamp column

possible_timestamp_cols = ["timestamp", "Timestamp", "time", "Time", "datetime", "Datetime", "date", "Date"]

timestamp_col = None

for col in possible_timestamp_cols:
    if col in df.columns:
        timestamp_col = col
        break

if timestamp_col is None:
    print("Available columns:", df.columns.tolist())
    raise ValueError("Timestamp column not detected. Please set timestamp_col manually.")

df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
invalid_timestamps = df[timestamp_col].isna().sum()

df = df.dropna(subset=[timestamp_col])
df = df.sort_values(timestamp_col).reset_index(drop=True)

print("Timestamp column:", timestamp_col)
print("Invalid timestamps removed:", invalid_timestamps)
print("Start:", df[timestamp_col].min())
print("End:", df[timestamp_col].max())
print("Shape after timestamp check:", df.shape)

## 4. Define documented failure events

These events are used to build the pre-failure warning labels.

In [ ]:
# 5. Define documented failure events

failure_events = pd.DataFrame({
    "event_id": ["F1", "F2", "F3", "F4"],
    "failure_type": ["Air leak", "Air leak", "Air leak", "Air leak"],
    "failure_start": [
        "2020-04-18 00:00",
        "2020-05-29 23:30",
        "2020-06-05 10:00",
        "2020-07-15 14:30"
    ],
    "failure_end": [
        "2020-04-18 23:59",
        "2020-05-30 06:00",
        "2020-06-07 14:30",
        "2020-07-15 19:00"
    ]
})

failure_events["failure_start"] = pd.to_datetime(failure_events["failure_start"])
failure_events["failure_end"] = pd.to_datetime(failure_events["failure_end"])

dataset_start = df[timestamp_col].min()
dataset_end = df[timestamp_col].max()

failure_events["inside_dataset_range"] = (
    (failure_events["failure_start"] >= dataset_start) &
    (failure_events["failure_end"] <= dataset_end)
)

failure_events.to_csv(OUTPUT_TABLES_DIR / "failure_events_for_label_creation.csv", index=False)

failure_events

## 5. Create failure interval column

Rows inside documented failure intervals are marked as `failure_interval = 1`.

They are not treated as normal records during supervised model training.

In [ ]:
# 6. Create failure interval label

df_labelled = df.copy()
df_labelled["failure_interval"] = 0
df_labelled["failure_event_id"] = ""

for _, event in failure_events.iterrows():
    mask_failure = (
        (df_labelled[timestamp_col] >= event["failure_start"]) &
        (df_labelled[timestamp_col] <= event["failure_end"])
    )

    df_labelled.loc[mask_failure, "failure_interval"] = 1
    df_labelled.loc[mask_failure, "failure_event_id"] = event["event_id"]

failure_count = int(df_labelled["failure_interval"].sum())
failure_percentage = failure_count / len(df_labelled) * 100

print("Rows inside documented failure intervals:", failure_count)
print("Failure interval percentage:", round(failure_percentage, 4), "%")

## 6. Create 6-hour, 12-hour and 24-hour warning labels

For each documented failure, observations before the failure start are labelled as warning cases.

The actual failure interval itself is not labelled as a pre-failure warning period.

In [ ]:
# 7. Create warning labels

warning_windows_hours = [6, 12, 24]

for hours in warning_windows_hours:
    label_col = f"warning_{hours}h"
    event_col = f"warning_{hours}h_event_id"

    df_labelled[label_col] = 0
    df_labelled[event_col] = ""

    for _, event in failure_events.iterrows():
        window_start = event["failure_start"] - pd.Timedelta(hours=hours)
        window_end = event["failure_start"]

        mask_warning = (
            (df_labelled[timestamp_col] >= window_start) &
            (df_labelled[timestamp_col] < window_end)
        )

        # Do not label actual failure interval as pre-failure warning.
        mask_warning = mask_warning & (df_labelled["failure_interval"] == 0)

        df_labelled.loc[mask_warning, label_col] = 1
        df_labelled.loc[mask_warning, event_col] = event["event_id"]

for hours in warning_windows_hours:
    label_col = f"warning_{hours}h"
    count = int(df_labelled[label_col].sum())
    pct = count / len(df_labelled) * 100
    print(f"{label_col}: {count} rows ({pct:.4f}%)")

## 7. Create model inclusion flag

`model_include = 1` means the row can be used for supervised early-warning modelling.

Rows inside actual failure intervals are excluded because the aim is pre-failure warning, not post-failure detection.

In [ ]:
# 8. Create model inclusion flag

df_labelled["model_include"] = np.where(df_labelled["failure_interval"] == 1, 0, 1)

print("Rows included for modelling:", int(df_labelled["model_include"].sum()))
print("Rows excluded as failure intervals:", int((df_labelled["model_include"] == 0).sum()))

## 8. Create failure window summary table

In [ ]:
# 9. Create failure window summary table

window_records = []

for _, event in failure_events.iterrows():
    record = {
        "event_id": event["event_id"],
        "failure_type": event["failure_type"],
        "failure_start": event["failure_start"],
        "failure_end": event["failure_end"],
        "inside_dataset_range": event["inside_dataset_range"]
    }

    for hours in warning_windows_hours:
        record[f"warning_{hours}h_start"] = event["failure_start"] - pd.Timedelta(hours=hours)
        record[f"warning_{hours}h_end"] = event["failure_start"] - pd.Timedelta(seconds=1)

    window_records.append(record)

failure_warning_windows = pd.DataFrame(window_records)
failure_warning_windows.to_csv(OUTPUT_TABLES_DIR / "failure_warning_windows.csv", index=False)

failure_warning_windows

## 9. Create class distribution tables

These tables show the severity of class imbalance for 6h, 12h and 24h warning labels.

In [ ]:
# 10. Create class distribution tables

class_distribution_records = []

model_df_temp = df_labelled[df_labelled["model_include"] == 1].copy()

for hours in warning_windows_hours:
    label_col = f"warning_{hours}h"

    counts = model_df_temp[label_col].value_counts().sort_index()
    total = len(model_df_temp)

    normal_count = int(counts.get(0, 0))
    warning_count = int(counts.get(1, 0))

    class_distribution_records.append({
        "label": label_col,
        "normal_count": normal_count,
        "warning_count": warning_count,
        "total_model_rows": total,
        "normal_percentage": normal_count / total * 100,
        "warning_percentage": warning_count / total * 100,
        "imbalance_ratio_normal_to_warning": normal_count / warning_count if warning_count > 0 else np.nan
    })

class_distribution = pd.DataFrame(class_distribution_records)
class_distribution.to_csv(OUTPUT_TABLES_DIR / "warning_label_class_distribution.csv", index=False)

class_distribution

## 10. Plot warning label imbalance

In [ ]:
# 11. Plot class imbalance for each warning horizon

plot_data = class_distribution[["label", "normal_count", "warning_count"]].copy()

x = np.arange(len(plot_data))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, plot_data["normal_count"], width, label="Normal")
plt.bar(x + width/2, plot_data["warning_count"], width, label="Warning")
plt.xticks(x, plot_data["label"])
plt.ylabel("Number of rows")
plt.title("Class distribution for early-warning labels")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / "warning_label_class_distribution.png", dpi=300)
plt.show()

## 11. Plot warning and failure timeline

This figure visualises the four documented failure events and their 12-hour warning windows.

In [ ]:
# 12. Plot 12-hour warning windows and failure intervals

plt.figure(figsize=(12, 4))

for idx, event in failure_events.iterrows():
    y = idx + 1

    warning_start = event["failure_start"] - pd.Timedelta(hours=12)
    warning_end = event["failure_start"]

    # Warning window
    plt.hlines(y, warning_start, warning_end, linewidth=8, label="12h warning window" if idx == 0 else "")

    # Failure interval
    plt.hlines(y, event["failure_start"], event["failure_end"], linewidth=8, label="Failure interval" if idx == 0 else "")

    plt.text(event["failure_start"], y + 0.12, event["event_id"], fontsize=10)

plt.yticks(range(1, len(failure_events) + 1), failure_events["event_id"])
plt.xlabel("Time")
plt.ylabel("Failure event")
plt.title("12-hour warning windows and documented failure intervals")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / "warning_windows_timeline_12h.png", dpi=300)
plt.show()

## 12. Save label creation summary

In [ ]:
# 13. Save label creation summary

label_creation_summary = pd.DataFrame({
    "item": [
        "Input rows",
        "Input columns",
        "Output rows",
        "Output columns",
        "Warning windows created",
        "Primary modelling target",
        "Failure events used",
        "Failure interval rows excluded from modelling",
        "Rows included for modelling",
        "6h warning rows",
        "12h warning rows",
        "24h warning rows"
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        df_labelled.shape[0],
        df_labelled.shape[1],
        "6h, 12h, 24h",
        "warning_12h",
        len(failure_events),
        int((df_labelled["model_include"] == 0).sum()),
        int(df_labelled["model_include"].sum()),
        int(df_labelled["warning_6h"].sum()),
        int(df_labelled["warning_12h"].sum()),
        int(df_labelled["warning_24h"].sum())
    ]
})

label_creation_summary.to_csv(OUTPUT_TABLES_DIR / "label_creation_summary.csv", index=False)

label_creation_summary

## 13. Save labelled dataset and 12-hour model-ready dataset

The full labelled dataset keeps all rows and label columns.

The model-ready dataset removes actual failure intervals and uses `warning_12h` as the primary target.

In [ ]:
# 14. Save labelled datasets

df_labelled.to_csv(labelled_path, index=False)

model_ready_df = df_labelled[df_labelled["model_include"] == 1].copy()
model_ready_df.to_csv(model_ready_path, index=False)

print("Full labelled dataset saved to:", labelled_path)
print("Full labelled shape:", df_labelled.shape)

print("\n12h model-ready dataset saved to:", model_ready_path)
print("Model-ready shape:", model_ready_df.shape)

## 14. Completion checklist

In [ ]:
# 15. Completion checklist

completion_checklist = pd.DataFrame({
    "task": [
        "Feature-engineered dataset loaded",
        "Failure events defined",
        "Failure interval column created",
        "6h warning label created",
        "12h warning label created",
        "24h warning label created",
        "Model inclusion flag created",
        "Failure warning windows table saved",
        "Class distribution table saved",
        "Warning distribution figure saved",
        "12h warning timeline figure saved",
        "Full labelled dataset saved",
        "12h model-ready dataset saved"
    ],
    "status": [
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete"
    ]
})

completion_checklist.to_csv(
    OUTPUT_TABLES_DIR / "label_creation_completion_checklist.csv",
    index=False
)

completion_checklist

## 15. Confirm saved outputs

In [ ]:
# 16. Confirm saved files

print("Saved label-related tables:")
for file in sorted(OUTPUT_TABLES_DIR.glob("*.csv")):
    if "label" in file.name or "warning" in file.name or "failure" in file.name:
        print("-", file.name)

print("\nSaved label-related figures:")
for file in sorted(OUTPUT_FIGURES_DIR.glob("*.png")):
    if "warning" in file.name or "label" in file.name:
        print("-", file.name)

print("\nProcessed labelled dataset files:")
for file in sorted(PROCESSED_DATA_DIR.glob("*labelled*")):
    print("-", file.name)

for file in sorted(PROCESSED_DATA_DIR.glob("*model_ready*")):
    print("-", file.name)